# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saurabh-kumar-ydv/FLYRANK-ML-WORKSPACE/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Actions are prioritized based on expected impact and urgency. Reason codes provide transparent context to content strategists to explain why an item requires action, ensuring trust in automated recommendations.

In [1]:
import pandas as pd

# Example implementation for generating a ranked action queue with reason codes
def generate_action_queue(df):
    # Define rules/reason codes based on features or predictions
    conditions = [
        (df["predicted_impact"] > 0.8) & (df["ctr"] < 0.02),
        (df["decay_score"] > 0.7),
        (df["bounce_rate"] > 0.75) & (df["time_on_page"] < 30),
    ]
    reason_codes = [
        "HIGH_POTENTIAL_CTR_GAP",
        "CONTENT_DECAY_REWRITE",
        "ENGAGEMENT_MISMATCH",
    ]

    df["reason_code"] = pd.np.select(
        conditions, reason_codes, default="ROUTINE_REVIEW"
    )
    df["priority_score"] = (
        df["predicted_impact"] * 0.6 + df["decay_score"] * 0.4
    )

    # Sort by priority
    action_queue = df.sort_values(
        by="priority_score", ascending=False
    ).reset_index(drop=True)
    return action_queue

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Designed for Content Operations, Editorial, and SEO Lead teams to prioritize monthly content updates, structural rewrites, title tag optimizations, and content refresh cycles.

Limits & Constraints:

This system serves as a decision-support tool, not an automated execution engine. Model recommendations lose accuracy during major search engine algorithm updates, unmodeled seasonal traffic shifts, or for newly published content (< 30 days live) lacking sufficient baseline analytics.

In [3]:
# Guardrail Check: Filter out new pages without enough baseline history (< 30 days)
MIN_DAYS_LIVE = 30

def apply_usage_limits(df):
    eligible_df = df[df["days_live"] >= MIN_DAYS_LIVE].copy()
    excluded_count = len(df) - len(eligible_df)
    print(f"Operational Limits Applied: Excluded {excluded_count} pages with < {MIN_DAYS_LIVE} days live.")
    return eligible_df

# Example execution:
# valid_df = apply_usage_limits(df)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Required:

Verification of search intent alignment before executing structural content rewrites.

Fact-checking, source verification, and regulatory/legal compliance checks.

No-Go List (Never Automated):

Auto-publishing copy edits directly to live URLs without editorial sign-off.

Bulk deletion, pruning, or URL redirection of core revenue-generating pages.

Changing canonical tags, site architecture, or URL paths based purely on automated model outputs.

In [7]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Load the actual dataset
# ============================================================

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# ============================================================
# 2. Safety Check
# ============================================================

# Inspect available content types first
print("\nAvailable content types:")
print(df["content_type"].value_counts(dropna=False))


# ============================================================
# 3. Define content types requiring manual review
# ============================================================

protected_content_types = [
    "legal",
    "transactional",
    "financial",
    "policy"
]

df["content_type_clean"] = (
    df["content_type"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


# ============================================================
# 4. Flag pages requiring manual approval
# ============================================================

df["requires_manual_approval"] = (
    df["content_type_clean"]
    .isin(protected_content_types)
)


# ============================================================
# 5. Count flagged pages
# ============================================================

flagged_count = int(
    df["requires_manual_approval"].sum()
)

print(
    f"\nFlagged {flagged_count} pages "
    f"for mandatory manual sign-off."
)


# ============================================================
# 6. Show flagged pages
# ============================================================

flagged_pages = df[
    df["requires_manual_approval"]
]

print("\nFlagged Pages:")

if len(flagged_pages) > 0:
    print(
        flagged_pages[
            [
                "content_id",
                "client_id",
                "content_type",
                "requires_manual_approval"
            ]
        ].head(20).to_string(index=False)
    )
else:
    print("No pages matched the protected content types.")

Dataset loaded successfully!
Dataset shape: (30000, 44)

Available content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Flagged 0 pages for mandatory manual sign-off.

Flagged Pages:
No pages matched the protected content types.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Recommendations are considered stale or invalid under the following conditions:

Data & Feature Drift: Statistical drift in input features (e.g., Population Stability Index (PSI) > 0.25) across traffic patterns or CTR distributions.

Performance Degradation: Model precision or rank correlation (NDCG / Precision@K) drops by > 15% relative to the baseline performance over a 30-day window.

Scheduled Cadence: Mandatory quarterly retraining cycle using refreshed search engine performance and organic traffic data.

In [8]:
import numpy as np

def check_retrain_triggers(baseline_precision, current_precision, psi_score, max_psi=0.25, max_drop=0.15):
    precision_drop = (baseline_precision - current_precision) / baseline_precision

    drift_detected = psi_score > max_psi
    perf_degraded = precision_drop > max_drop

    triggers = []
    if drift_detected:
        triggers.append(f"Feature Drift Detected (PSI: {psi_score:.2f} > {max_psi})")
    if perf_degraded:
        triggers.append(f"Performance Drop Detected ({precision_drop:.1%} drop > {max_drop:.0%})")

    retrain_required = drift_detected or perf_degraded
    print(f"Retrain Triggered: {retrain_required}")
    if triggers:
        print("Reasons:", ", ".join(triggers))

    return retrain_required



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Outputs and prioritized queues are exported to work/outputs/ to ensure downstream paper generation and operational dashboards pull consistent, reproducible data.

In [9]:
import os

# Ensure directory exists and save output queue
os.makedirs("../outputs", exist_ok=True)
output_path = "../outputs/action_queue.csv"

# Uncomment when action_queue DataFrame is generated:
# action_queue.to_csv(output_path, index=False)
print(f"Successfully configured export path: {output_path}")

Successfully configured export path: ../outputs/action_queue.csv


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.